# Data audit and preregistered protocol
We ask whether DNH accuracy changes during a stream, relative to adapting ARIMA, and how different non-stationarities affect adaptation. One customer, one stream; forecast the next hourly consumption from past targets only. No pooling or concatenation.

**Source verification:** the published environment now permits retrieval. Electricity record 4656140 version 3 and its archive were verified against the source checksum. The original DNH full text and XML were retrieved; implementation ambiguities are explicitly resolved for a separate DNH-inspired v1; faithful DNH-HOPE remains unimplemented. Consult `docs/METHOD.md` and `sources/access_retry.json`.

Quick: 168 development + 24 evaluation points, up to 5 series. Pilot: 720 + 168, 5 series. Full: 2160 + 2160, 10 series. Full and pilot require explicit configuration. These are bounded chronological prefixes, not a claim to cover all three years. The appended experiments evaluate DNH-inspired v1, not faithful DNH. See docs/DNH_INSPIRED.md for frozen assumptions and limits.

In [ ]:
from pathlib import Path
import os, sys, json, time
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'research').is_dir())
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
os.environ.setdefault('OMP_NUM_THREADS','1')
os.environ.setdefault('OPENBLAS_NUM_THREADS','1')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from research.experiment import config, hardware, run_stream, metrics
from research.data import acquire, parse_tsf, audit, select
from research.synthetic import processes, change_analysis
from research.inspired_experiment import run_inspired, inspired_config
from research.inspired_study import compare_streams, controlled
from research.validation import source, electricity_source
MODE = os.environ.get('EXPERIMENT_MODE','quick')
cfg = config(MODE)
OUT = Path(os.environ.get('DNH_RESULTS_DIR',ROOT/'results'/MODE))
OUT.mkdir(parents=True, exist_ok=True)
print(hardware()); print(cfg)
def stream_provenance(sid, values, generator_seed=17):
    if series:
        return electricity_source({'id':sid,'values':values},verified)
    return source('synthetic',generator_seed=generator_seed,process='stationary')


## Download or manual cache
Run `python scripts/fetch_data.py` after network access is available. It verifies the archive against record metadata and records SHA256, source checksum, URL, version (or absent), retrieval time, and ZIP members. For offline use, download the record API JSON as `data/raw/record.json` and the original ZIP beside it. Metadata and archive must match. Notebook execution never silently downloads or invents electricity data.

In [ ]:
archive = sorted((ROOT/'data/raw').glob('*.zip'))
series = []
if archive:
    verified = acquire(ROOT/'data/raw', manual=True)
    metadata, series = parse_tsf(verified)
    print(metadata)
    data_audit = audit(series)
    display(data_audit)
    data_audit.to_csv(OUT/'electricity_audit.csv', index=False)
    print('Actual series count:', len(series), 'lengths:', data_audit.length.value_counts().to_dict())
    print('Claim checks (not assumed):', {'321_series':len(series)==321,
          'all_start_2012':data_audit.start.str.startswith('2012').all(),
          'all_end_2014':data_audit.end.str.startswith('2014').all()})
else:
    print('ELECTRICITY BLOCKED: no source-verified archive. No electricity results generated.')


## Audit and selection
Preserve missing observations and zeros. No interpolation, winsorization, or normalization. The initial development segment must be finite; evaluation missing targets remain unscored. Persistence and seasonal naive may emit missing predictions when their lag is missing; report denominators and scored counts. Flag IQR extremes, do not replace them. TSF grids derived from a declared start/frequency cannot reveal irregularities omitted in the original aggregation.

Choose lexicographically sorted series IDs with sufficient length and finite development history, before examining forecasting performance. Choose bounded ARIMA orders by converged development AIC only. Evaluation starts at the development boundary; all models predict then receive each target. ARIMA uses expanding history, updates its filtered state each observation and refits the selected order on the configured schedule. Persistence and seasonal naive require only their fixed lags; all have access to the same target history. No extra covariates.

MASE uses development lag-24 differences only. Zero denominator means undefined MASE, not zero error. No random split. AIC comparison across differencing orders is a pragmatic bounded selector, not evidence of optimality; consider a development-only chronological validation extension for a final study.

In [ ]:
chosen = select(series,cfg) if series else []
for s in chosen:
    y=s['values']; n=cfg['development']
    fig, ax=plt.subplots(figsize=(10,3))
    ax.plot(s['index'][:n+cfg['steps']],y[:n+cfg['steps']],label='observed')
    ax.plot(s['index'][:n+cfg['steps']],pd.Series(y[:n+cfg['steps']]).rolling(24).mean(),label='trailing mean')
    ax.axvline(s['index'][n],color='k',ls='--'); ax.legend(); ax.set_title(s['id'])
    fig.savefig(OUT/f"audit_{s['id']}.png"); plt.show()
print('Selected IDs:',[s['id'] for s in chosen])

Seasonality is predictable repetition; outliers are unusual individual observations. Changes in the conditional forecasting relationship alter how history predicts the next target. A changing marginal mean or rolling variance alone does not establish such a change. Natural data associations cannot identify causal drift.